# ATML PA2 GPU runner (Google Colab)

This notebook checks out the public assignment repository, installs its pinned dependencies, downloads the fixed course assets, and runs the documented experiments. It does not generate report prose. The assignment requires the student to write and interpret the final report.

Before running: in Colab choose **Runtime → Change runtime type → T4 GPU** (or a stronger available GPU). Run cells in order. The full suite is long and may exceed one session; trained adapters and results are stored in Google Drive.


## 1. Check out the implementation

This uses the public fork so the run corresponds to a published commit.


In [ ]:
%cd /content
!git clone https://github.com/ttqureshi/ATML-PA2-LLM-PostTraining.git
%cd /content/ATML-PA2-LLM-PostTraining
!git rev-parse HEAD


## 2. Confirm a GPU is attached

Stop here if CUDA is unavailable. The assignment's full training and evaluation workloads are not practical on CPU.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Colab select Runtime > Change runtime type > T4 GPU, reconnect, and rerun this cell.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}; VRAM: {props.total_memory / 2**30:.1f} GiB")


## 3. Install the pinned environment and persist outputs

The repository pins Transformers, TRL, PEFT, and tokenizer versions. The existing Colab CUDA-enabled PyTorch satisfies the `torch>=2.4` requirement. Google Drive stores adapters and result logs across runtime resets.


In [ ]:
%pip install -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
repo = Path("/content/ATML-PA2-LLM-PostTraining")
persist = Path("/content/drive/MyDrive/ATML-PA2-LLM-PostTraining")
persist.mkdir(parents=True, exist_ok=True)
for name in ("outputs", "results"):
    source = repo / name
    destination = persist / name
    destination.mkdir(parents=True, exist_ok=True)
    if source.exists() and not source.is_symlink():
        shutil.copytree(source, destination, dirs_exist_ok=True)
        shutil.rmtree(source)
    if not source.is_symlink():
        source.symlink_to(destination, target_is_directory=True)
print("Persistent experiment files:", persist)


## 4. Download and validate the fixed course assets

This fetches the pinned release (about 1 GB) and verifies each pinned asset and the expected schemas.


In [ ]:
!python -m scripts.check_environment
!python -m scripts.download_assets
!python -m scripts.validate_assets


## 5. Task 1 — DPO

Run the one-epoch standard policy, its held-out evaluation, the three beta forks, and the separate length-balanced study.


In [ ]:
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml
!python -m task1_dpo.analyze_length --config configs/dpo.yaml


## 6. Task 2 — PPO

Continue from the supplied policy and value midpoint, then run cached clipping analysis and the clipping and KL forks.


In [ ]:
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml
!python -m task2_ppo.ablate_kl --config configs/ppo.yaml


## 7. Task 3 — GRPO

Continue from the supplied midpoint, evaluate, regroup the fixed cache at K=2/4/8, then run both normalization forks.


In [ ]:
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard
!python -m task3_grpo.analyze_group_size --config configs/grpo.yaml
!python -m task3_grpo.compare_normalization --config configs/grpo.yaml


## 8. Task 4 — Safety calibration

Generate responses for the four required policies and create the 60-prompt audit sheet. Complete every manual label yourself before opening the AI-judge results. The notebook does not fabricate manual labels.


In [ ]:
!python -m task4_safety.generate_responses --config configs/feedback.yaml
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml


After the previous cell completes, edit `results/task4_safety/manual_audit.csv` in Google Drive. Label 30 safe and 30 unsafe prompts for each policy response using the required categories. Then resume and run:


In [ ]:
!python -m task4_safety.judge_responses --config configs/feedback.yaml
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml


## 9. Task 5 — RLVR versus RLAIF

Evaluate the fixed supplied policies on GSM8K and transfer, score the controlled diagnostics, then combine the saved comparisons.


In [ ]:
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm
!python -m task5_feedback.score_perturbations --config configs/feedback.yaml
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer
!python -m task5_feedback.compare_feedback --config configs/feedback.yaml


## 10. Check saved artifacts

Review the JSON/JSONL/CSV files under `MyDrive/ATML-PA2-LLM-PostTraining/results`. Keep generated model outputs and the exact repository commit with the experiment logs. You still need to inspect examples, complete the human safety audit, and write the report in your own words as the assignment requires.


In [ ]:
!git -C /content/ATML-PA2-LLM-PostTraining rev-parse HEAD
!find /content/drive/MyDrive/ATML-PA2-LLM-PostTraining/results -maxdepth 2 -type f | sort | head -80
